# Lab 2 · Track W — Weather Station training notebook

**EECE.4862 / 5862 — Embedded Artificial Intelligence**

Run this notebook **top to bottom** in Google Colab (or local Python 3 +
TensorFlow). It loads the bundled weather dataset, builds a 3-hour
temperature/humidity feature window, trains a small snowfall classifier,
**quantizes it to int8**, and exports it as `model.cpp` for the
`w2-weather-inference` sketch. No weather-API key and no external download
are needed — the dataset ships next to this notebook
(`weather-dataset.csv`).

You are not building the pipeline from scratch: you **run it**, read the
numbers it prints, and make the **one required change** in the last
section so your report reflects a run you actually drove.

## Dataset provenance

`weather-dataset.csv` is a frozen snapshot of **Environment and Climate
Change Canada (ECCC)** historical *hourly* climate observations for the
year **2012** (8784 hourly rows, no gaps), redistributed under the **Open
Government Licence – Canada**. Columns: `datetime, temp_c, humidity_pct,
weather, snow`, where `snow = 1` when the hour's `weather` text contains
"Snow". Snow is the minority class (~6.6% of hours), so we use class
weights below — a realistic class-imbalance situation you will reflect on
in §B.3.

In [1]:
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
np.random.seed(42); tf.random.set_seed(42)

# In Colab: upload weather-dataset.csv (it ships in lab-2/training/) if it
# is not already in the working directory.
try:
    df = pd.read_csv('weather-dataset.csv')
except FileNotFoundError:
    from google.colab import files
    files.upload()  # choose weather-dataset.csv
    df = pd.read_csv('weather-dataset.csv')

print(df.shape)
print(df.head())
print('snow hours:', int(df.snow.sum()), f'({100*df.snow.mean():.1f}%)')

Saving weather-dataset.csv to weather-dataset.csv
(8784, 5)
           datetime  temp_c  humidity_pct               weather  snow
0  2012-01-01 00:00    -1.8          86.0                   Fog     0
1  2012-01-01 01:00    -1.8          87.0                   Fog     0
2  2012-01-01 02:00    -1.8          89.0  Freezing Drizzle,Fog     0
3  2012-01-01 03:00    -1.5          88.0  Freezing Drizzle,Fog     0
4  2012-01-01 04:00    -1.5          88.0                   Fog     0
snow hours: 583 (6.6%)


## 1. Build the feature window

Each example is the last **`kWindowLen`** hourly `(temperature, humidity)`
readings; the label is whether it snows in the **next** hour. With hourly
data, `kWindowLen = 3` means "the last 3 hours" — exactly the handout
wording.

> **Copy the printed `kWindowLen` into `w2-weather-inference.ino`.**

In [2]:
kWindowLen = 6            # 3 hourly steps == 'last 3 hours'
kFeaturesPerStep = 2      # temperature + humidity
print('kWindowLen =', kWindowLen)

t = df['temp_c'].to_numpy(); h = df['humidity_pct'].to_numpy(); s = df['snow'].to_numpy()
X, y = [], []
for i in range(len(df) - kWindowLen):
    win = []
    for k in range(kWindowLen):
        win += [t[i+k], h[i+k]]
    X.append(win); y.append(s[i + kWindowLen])
X = np.array(X, dtype=np.float32); y = np.array(y, dtype=np.int32)
print('X', X.shape, 'y', y.shape, 'positives', int(y.sum()))

kWindowLen = 6
X (8778, 12) y (8778,) positives 583


## 2. Split + standardize

We standardize each feature (subtract mean, divide by std) using stats
from the **training split only**. The device must apply the identical
transform.

> **Copy the four printed constants into `normTemp()` / `normHumidity()`
> in the sketch.**

In [3]:
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
mean_t, std_t = float(Xtr[:,0::2].mean()), float(Xtr[:,0::2].std())
mean_h, std_h = float(Xtr[:,1::2].mean()), float(Xtr[:,1::2].std())
print(f'mean_t={mean_t:.6f} std_t={std_t:.6f}')
print(f'mean_h={mean_h:.6f} std_h={std_h:.6f}')

def standardize(A):
    A = A.copy()
    A[:,0::2] = (A[:,0::2] - mean_t) / std_t
    A[:,1::2] = (A[:,1::2] - mean_h) / std_h
    return A
Xtr_n, Xva_n = standardize(Xtr), standardize(Xva)

mean_t=8.718309 std_t=11.692684
mean_h=67.345032 std_h=16.831001


## 3. Train a small classifier

A tiny dense network (Lab-1 difficulty). Class weights counter the snow
imbalance so the model doesn't trivially predict "no snow" every time.

In [4]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(kWindowLen*kFeaturesPerStep,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(8,  activation='relu'),
    tf.keras.layers.Dense(1,  activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
n0, n1 = (ytr==0).sum(), (ytr==1).sum()
cw = {0: len(ytr)/(2*n0), 1: len(ytr)/(2*n1)}
hist = model.fit(Xtr_n, ytr, validation_data=(Xva_n, yva),
                 epochs=40, batch_size=64, class_weight=cw, verbose=2)
print('final train_acc=%.4f  val_acc=%.4f' % (hist.history['accuracy'][-1], hist.history['val_accuracy'][-1]))

Epoch 1/40
110/110 - 2s - 16ms/step - accuracy: 0.8375 - loss: 0.6312 - val_accuracy: 0.7842 - val_loss: 0.4027
Epoch 2/40
110/110 - 0s - 3ms/step - accuracy: 0.7079 - loss: 0.4293 - val_accuracy: 0.7449 - val_loss: 0.4185
Epoch 3/40
110/110 - 0s - 3ms/step - accuracy: 0.7350 - loss: 0.3773 - val_accuracy: 0.7682 - val_loss: 0.3922
Epoch 4/40
110/110 - 0s - 3ms/step - accuracy: 0.7585 - loss: 0.3621 - val_accuracy: 0.7751 - val_loss: 0.3825
Epoch 5/40
110/110 - 0s - 3ms/step - accuracy: 0.7664 - loss: 0.3538 - val_accuracy: 0.7802 - val_loss: 0.3788
Epoch 6/40
110/110 - 0s - 3ms/step - accuracy: 0.7700 - loss: 0.3482 - val_accuracy: 0.7825 - val_loss: 0.3765
Epoch 7/40
110/110 - 0s - 3ms/step - accuracy: 0.7723 - loss: 0.3439 - val_accuracy: 0.7864 - val_loss: 0.3694
Epoch 8/40
110/110 - 0s - 3ms/step - accuracy: 0.7761 - loss: 0.3405 - val_accuracy: 0.7893 - val_loss: 0.3683
Epoch 9/40
110/110 - 0s - 3ms/step - accuracy: 0.7803 - loss: 0.3368 - val_accuracy: 0.7938 - val_loss: 0.3640


In [5]:
pv = (model.predict(Xva_n, verbose=0).ravel() > 0.5).astype(int)
print('confusion [[TN FP][FN TP]]:')
print(confusion_matrix(yva, pv))
print('val accuracy = %.4f' % accuracy_score(yva, pv))

confusion [[TN FP][FN TP]]:
[[1301  338]
 [  10  107]]
val accuracy = 0.8018


## 4. Quantize to int8

Full int8 quantization with a representative dataset from the train
split — the same recipe Lab 1's hello-world model used. Record the
float-vs-int8 sizes and accuracies (useful for the 5862 Part C.1 option).

In [6]:
def rep_data():
    for i in range(min(500, len(Xtr_n))):
        yield [Xtr_n[i:i+1].astype(np.float32)]
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep_data
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
tfl_int8 = conv.convert()
print('int8 tflite bytes =', len(tfl_int8))

# verify int8 accuracy on the validation set
itp = tf.lite.Interpreter(model_content=tfl_int8); itp.allocate_tensors()
ind = itp.get_input_details()[0]; outd = itp.get_output_details()[0]
isc, izp = ind['quantization']; osc, ozp = outd['quantization']
print(f'input scale={isc:.8f} zp={izp}  output scale={osc:.8f} zp={ozp}')
qp = []
for row in Xva_n:
    q = np.round(row/isc + izp).astype(np.int8).reshape(ind['shape'])
    itp.set_tensor(ind['index'], q); itp.invoke()
    o = int(itp.get_tensor(outd['index']).ravel()[0])
    qp.append(1 if (o - int(ozp))*osc > 0.5 else 0)
print('int8 val accuracy = %.4f' % accuracy_score(yva, np.array(qp)))

Saved artifact at '/tmp/tmps73to7vg'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 12), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134128257673616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134128257674768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134128257673424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134128257672080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134128257675344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134128257672272: TensorSpec(shape=(), dtype=tf.resource, name=None)
int8 tflite bytes = 3376
input scale=0.01824300 zp=13  output scale=0.00390625 zp=-128
int8 val accuracy = 0.8144


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## 5. Export `model.cpp`

Writes the int8 model as a C byte array (`g_model` / `g_model_len`), the
same form as Lab 1's hello-world `model.cpp`. In Colab, the last line
downloads the file; drop it into `w2-weather-inference/` next to the
sketch (replacing the placeholder).

In [7]:
def to_c_array(data):
    lines = [f'  0x{b:02x},' for b in data]
    body = ''
    for i in range(0, len(lines), 12):
        body += ''.join(lines[i:i+12]).rstrip() + '\n'
    return body

with open('model.cpp', 'w') as f:
    f.write('// Lab 2 - Track W - model.cpp  (exported from weather-training.ipynb)\n')
    f.write('#include "model.h"\n\n')
    f.write('alignas(8) const unsigned char g_model[] = {\n')
    f.write(to_c_array(tfl_int8))
    f.write('};\n')
    f.write('const int g_model_len = sizeof(g_model);\n')
print('wrote model.cpp (', len(tfl_int8), 'bytes )')
try:
    from google.colab import files; files.download('model.cpp')
except Exception:
    pass

wrote model.cpp ( 3376 bytes )


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Copy the constants into the sketch

Before flashing `w2-weather-inference.ino`, make sure these match what
you printed above:

```cpp
constexpr int kWindowLen = 3;
constexpr float kMeanT = 8.736408f, kStdT = 11.688525f;
constexpr float kMeanH = 67.333000f, kStdH = 16.830614f;
```

(The input quantization scale/zero-point are read from the model at
runtime, so you do not copy those.)

## 7. ⭐ Required change (do this, then re-run)

Per handout §B.1, make **one** change so your submission reflects a run
you drove, then re-run the notebook top to bottom and note the effect in
your report:

- **Easiest:** change `kWindowLen` in §1 (e.g. to `1`, `6`, or `12`) and
  compare validation accuracy and `model.cpp` size to the default 3. (If
  you change it, update `kWindowLen` in the sketch to match!)
- **Or:** change the `test_size` / `random_state` of the split in §2 and
  report how validation accuracy moves.

Write 2–3 sentences in your report: what you changed, and what happened
to accuracy and model size.